In [1]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0) File settings
# ============================================================

INPUT_FILE = "upu.xlsx"                 
SHEET_NAME = 0                          
OUTPUT_FILE = "weekly_trends_sg.xlsx"   

# ============================================================
# 1) Basic transformation helpers
# ============================================================

def parse_tr_date_series(s: pd.Series) -> pd.Series:
  
    dt = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    if dt.isna().any():
        dt2 = pd.to_datetime(s, dayfirst=True, errors="coerce")
        dt = dt.fillna(dt2)
    return dt


def to_float_tr(s: pd.Series) -> pd.Series:
   
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")

    ser = s.astype(str).str.strip()
    ser = ser.replace({"": np.nan, "nan": np.nan, "None": np.nan, "NaT": np.nan})

    has_dot = ser.str.contains(r"\.", regex=True, na=False)
    has_comma = ser.str.contains(",", regex=False, na=False)
    mask_thousands = has_dot & has_comma
    ser.loc[mask_thousands] = ser.loc[mask_thousands].str.replace(".", "", regex=False)

    ser = ser.str.replace(",", ".", regex=False)
    return pd.to_numeric(ser, errors="coerce")


def week_monday(d: pd.Timestamp) -> pd.Timestamp:
    """Returns the Monday of the week for a given date."""
    return d - pd.to_timedelta(d.weekday(), unit="D")


# ============================================================
# 2) Sklansky–Gonzalez (ε-tolerance) 
# ============================================================

def sklansky_gonzalez_piecewise(
    x: np.ndarray,
    y: np.ndarray,
    eps: float,
    *,
    choose_slope: str = "mid"  
):
    n = len(x)
    if n < 2:
        return []

    segments = []
    s = 0

    while s < n - 1:
        xs, ys = float(x[s]), float(y[s])
        m_min, m_max = -np.inf, np.inf
        k = s + 1

        last_valid_m_min, last_valid_m_max = m_min, m_max
        last_valid_e = s + 1

        while k < n:
            dx = float(x[k] - xs)
            if dx == 0:
                k += 1
                continue

            L = (float(y[k]) - eps - ys) / dx
            U = (float(y[k]) + eps - ys) / dx

            new_m_min = max(m_min, L)
            new_m_max = min(m_max, U)

            if new_m_min > new_m_max:
                break

            m_min, m_max = new_m_min, new_m_max
            last_valid_m_min, last_valid_m_max = m_min, m_max
            last_valid_e = k
            k += 1

        e = last_valid_e

        if choose_slope == "min":
            m = last_valid_m_min
        elif choose_slope == "max":
            m = last_valid_m_max
        else:
            m = (last_valid_m_min + last_valid_m_max) / 2.0

        b = ys - m * xs
        segments.append({"s": int(s), "e": int(e), "m": float(m), "b": float(b)})

        if e <= s:
            break

        s = e

    return segments


# ============================================================
# 3)Optional machine-based eps
# ============================================================

def compute_machine_eps(
    df_machine: pd.DataFrame,
    value_col: str,
    *,
    c: float = 1.5,
    min_eps: float = 0.5,
    max_eps: float = 10.0
) -> float:
    y = df_machine[value_col].dropna().values.astype(float)
    if len(y) < 3:
        return float(min_eps)

    diffs = np.abs(np.diff(y))
    med = float(np.median(diffs)) if len(diffs) else 0.0
    eps = c * med
    eps = max(min_eps, eps)
    eps = min(max_eps, eps)
    return float(eps)


# ============================================================
# 4) Main pipeline: weekly (Mon–Fri) trend
# ============================================================

def weekly_trends_sg(
    upu: pd.DataFrame,
    *,
    date_col: str = "Date",
    machine_col: str = "Machine_ID",
    value_col: str = "OEE",
    scale: str = "0-100",
    eps_mode: str = "fixed",     
    fixed_eps: float = 2.0,
    per_machine_c: float = 1.5,
    per_machine_min_eps: float = 0.5,
    per_machine_max_eps: float = 10.0,
    choose_slope: str = "mid",
    required_days_in_week: int = 5
) -> pd.DataFrame:

    df = upu.copy()

    df[date_col] = parse_tr_date_series(df[date_col])
    df[value_col] = to_float_tr(df[value_col])

    df = df.dropna(subset=[date_col, machine_col, value_col]).copy()

    if scale == "0-1":
        df[value_col] = df[value_col] / 100.0
        fixed_eps = fixed_eps / 100.0
        per_machine_min_eps = per_machine_min_eps / 100.0
        per_machine_max_eps = per_machine_max_eps / 100.0


    df = df[df[date_col].dt.weekday <= 4].copy()

    
    df["WeekStart"] = df[date_col].apply(week_monday)


    df = (
        df.groupby([machine_col, date_col, "WeekStart"], as_index=False)[value_col]
          .mean()
          .sort_values([machine_col, date_col])
    )

    eps_by_machine = {}
    if eps_mode == "per_machine":
        for mid, g in df.groupby(machine_col):
            g_sorted = g.sort_values(date_col)
            eps_by_machine[mid] = compute_machine_eps(
                g_sorted, value_col,
                c=per_machine_c,
                min_eps=per_machine_min_eps,
                max_eps=per_machine_max_eps
            )

    rows = []

    for mid, dfg in df.groupby(machine_col):
        dfg = dfg.sort_values(["WeekStart", date_col]).copy()
        week_starts = sorted(dfg["WeekStart"].unique())

        for ws in week_starts:
            prev_ws = ws - pd.Timedelta(days=7)

            g_week = dfg[dfg["WeekStart"] == ws].sort_values(date_col)
            g_prev = dfg[dfg["WeekStart"] == prev_ws].sort_values(date_col)

            cur = len(g_week) if len(g_week) else None
            prev = len(g_prev) if len(g_prev) else None

            if cur != required_days_in_week or prev != required_days_in_week:
                continue

            x = np.arange(len(g_week), dtype=float)  
            y = g_week[value_col].values.astype(float)

            eps = eps_by_machine[mid] if eps_mode == "per_machine" else fixed_eps
            segs = sklansky_gonzalez_piecewise(x, y, eps, choose_slope=choose_slope)
            if not segs:
                continue

            slopes = [s["m"] for s in segs]
            breaks = [s["e"] for s in segs[:-1]]
            break_dates = [g_week.iloc[i][date_col] for i in breaks] if breaks else []

            seg_lengths = [max(0, int(s["e"]) - int(s["s"])) for s in segs]
            denom = float(sum(seg_lengths))
            if denom > 0:
                m_hafta = float(sum(mi * Li for mi, Li in zip(slopes, seg_lengths)) / denom)
            else:
                m_hafta = np.nan

            seg_n_points = []
            seg_mae = []
            seg_rmse = []
            seg_maxabs = []
            seg_mae_over_eps = []
            seg_rmse_over_eps = []
            seg_maxabs_over_eps = []

            total_points = 0
            sum_abs = 0.0
            sum_sq = 0.0
            week_maxabs = 0.0

            eps_safe = float(eps) if float(eps) != 0.0 else np.nan

            for j, seg in enumerate(segs):
                s_idx = int(seg["s"])
                e_idx = int(seg["e"])

              
                s_idx = max(0, min(s_idx, len(x) - 1))
                e_idx = max(0, min(e_idx, len(x) - 1))
                if e_idx < s_idx:
                    s_idx, e_idx = e_idx, s_idx

                if j > 0:
                    s_idx = s_idx + 1

                if s_idx > e_idx:
                    seg_n_points.append(0)
                    seg_mae.append(np.nan)
                    seg_rmse.append(np.nan)
                    seg_maxabs.append(np.nan)
                    seg_mae_over_eps.append(np.nan)
                    seg_rmse_over_eps.append(np.nan)
                    seg_maxabs_over_eps.append(np.nan)
                    continue

                idx = np.arange(s_idx, e_idx + 1, dtype=int)

                m = float(seg["m"])
                b = float(seg["b"])
                yhat = m * x[idx] + b
                resid = y[idx] - yhat

                abs_resid = np.abs(resid)
                mae = float(np.mean(abs_resid))
                rmse = float(np.sqrt(np.mean(resid ** 2)))
                mx = float(np.max(abs_resid))

                seg_n_points.append(int(len(idx)))
                seg_mae.append(mae)
                seg_rmse.append(rmse)
                seg_maxabs.append(mx)

                if np.isnan(eps_safe):
                    seg_mae_over_eps.append(np.nan)
                    seg_rmse_over_eps.append(np.nan)
                    seg_maxabs_over_eps.append(np.nan)
                else:
                    seg_mae_over_eps.append(float(mae / eps_safe))
                    seg_rmse_over_eps.append(float(rmse / eps_safe))
                    seg_maxabs_over_eps.append(float(mx / eps_safe))

                total_points += int(len(idx))
                sum_abs += float(np.sum(abs_resid))
                sum_sq += float(np.sum(resid ** 2))
                week_maxabs = max(week_maxabs, mx)

            if total_points > 0:
                week_mae = float(sum_abs / total_points)
                week_rmse = float(np.sqrt(sum_sq / total_points))
                week_maxabs = float(week_maxabs)
            else:
                week_mae = np.nan
                week_rmse = np.nan
                week_maxabs = np.nan

            if np.isnan(eps_safe):
                week_mae_over_eps = np.nan
                week_rmse_over_eps = np.nan
                week_maxabs_over_eps = np.nan
            else:
                week_mae_over_eps = float(week_mae / eps_safe) if not np.isnan(week_mae) else np.nan
                week_rmse_over_eps = float(week_rmse / eps_safe) if not np.isnan(week_rmse) else np.nan
                week_maxabs_over_eps = float(week_maxabs / eps_safe) if not np.isnan(week_maxabs) else np.nan

            rows.append({
                "Machine_ID": mid,
                "WeekStart": ws,
                "PrevWeekStart": prev_ws,
                "DaysCount": int(len(g_week)),
                "PrevDaysCount": int(len(g_prev)),
                "EpsUsed": float(eps),
                "SegmentsCount": int(len(segs)),
                "Slopes": slopes,
                "SegLengths": seg_lengths,
                "m_hafta": m_hafta,
                "BreakIdx": breaks,
                "BreakDates": break_dates,

        
                "SegNPoints": seg_n_points,
                "SegMAE": seg_mae,
                "SegRMSE": seg_rmse,
                "SegMaxAbs": seg_maxabs,
                "SegMAE_over_eps": seg_mae_over_eps,
                "SegRMSE_over_eps": seg_rmse_over_eps,
                "SegMaxAbs_over_eps": seg_maxabs_over_eps,

                "WeekMAE": week_mae,
                "WeekRMSE": week_rmse,
                "WeekMaxAbs": week_maxabs,
                "WeekMAE_over_eps": week_mae_over_eps,
                "WeekRMSE_over_eps": week_rmse_over_eps,
                "WeekMaxAbs_over_eps": week_maxabs_over_eps,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["LastSlope"] = out["Slopes"].apply(
        lambda lst: float(lst[-1]) if isinstance(lst, list) and len(lst) else np.nan
    )

    out = out.sort_values(["Machine_ID", "WeekStart"]).reset_index(drop=True)
    return out


# ============================================================
# 5) Read from Excel -> extract trend -> print -> save to Excel
# ============================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Excel not found: {INPUT_FILE}. "
        "Please ensure the file is in the same folder as the code."
    )

upu = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

weekly = weekly_trends_sg(
    upu,
    value_col="OEE",
    scale="0-100",
    eps_mode="fixed",          
    fixed_eps=5.0,
    choose_slope="mid",
    required_days_in_week=5
)

print("weekly satır sayısı (current=5 ve prev=5 olan haftalar):", len(weekly))
print(weekly.head(20))

weekly.to_excel(OUTPUT_FILE, index=False)
print(f"Excel kaydedildi: {OUTPUT_FILE}")


weekly satır sayısı (current=5 ve prev=5 olan haftalar): 225
           Machine_ID  WeekStart PrevWeekStart  DaysCount  PrevDaysCount  \
0   BIGLIA B 301 SM 1 2025-07-28    2025-07-21          5              5   
1   BIGLIA B 301 SM 1 2025-08-04    2025-07-28          5              5   
2   BIGLIA B 301 SM 1 2025-08-11    2025-08-04          5              5   
3   BIGLIA B 301 SM 1 2025-08-18    2025-08-11          5              5   
4   BIGLIA B 301 SM 1 2025-09-15    2025-09-08          5              5   
5   BIGLIA B 301 SM 1 2025-09-22    2025-09-15          5              5   
6   BIGLIA B 301 SM 1 2025-09-29    2025-09-22          5              5   
7   BIGLIA B 301 SM 1 2025-11-10    2025-11-03          5              5   
8   BIGLIA B 301 SM 1 2025-11-17    2025-11-10          5              5   
9   BIGLIA B 301 SM 1 2025-12-08    2025-12-01          5              5   
10  BIGLIA B 301 SM 1 2025-12-15    2025-12-08          5              5   
11  BIGLIA B 301 SM 1 2025-

Excel kaydedildi: weekly_trends_sg.xlsx
